# Phase 3
## Feature Engineering in Fraud Detection ML
 Handling Class Imbalance and fixing it wit SMOTE (Synthetic Minority Over-sampling Technique)

In [ ]:
# Importing libraries packages
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load your preprocessed splits from Phase 2
X_train = pd.read_csv('preprocessed/X_train.csv')
X_test  = pd.read_csv('preprocessed/X_test.csv')
y_train = pd.read_csv('preprocessed/y_train.csv').squeeze()
y_test  = pd.read_csv('preprocessed/y_test.csv').squeeze()

print("Loaded preprocessed data.")
print(f"X_train:  {X_train.shape}, fraud rate:  {y_train.mean()*100:.4f}%")

In [ ]:
# Understanding what we have before SMOTE
print("Class distribution BEFORE SMOTE:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).mul(100).round(4))


In [ ]:
# Applying SMOTE to training Data only
smote = SMOTE(
    sampling_strategy='auto',   # Oversample minority class to match majority
    k_neighbors=5,              # How many neighbors to interpolate between
    random_state=42             # Reproducibility
)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# After SMOTE: class counts
print("Class distribution AFTER SMOTE:")
print(pd.Series(y_train_res).value_counts())
print(pd.Series(y_train_res).value_counts(normalize=True).mul(100).round(2))

print(f"\nX_train before SMOTE: {X_train.shape}")
print(f"X_train after SMOTE:    {X_train_res.shape}")


In [ ]:
# Visualize Class Balance before and after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# before
before_counts = y_train.value_counts()
axes[0].bar(['Normal', 'Fraud'],
            [before_counts[0], before_counts[1]],
            color=['#378ADD', '#E24B4A'])
axes[0].set_title('Before SMOTE')
axes[0].set_ylabel('Count')
for i, v in enumerate([before_counts[0], before_counts[1]]):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

# After
after_counts = pd.Series(y_train_res).value_counts()
axes[1].bar(['Normal', 'Fraud (incl. synthetic)'],
            [after_counts[0], after_counts[1]],
            color=['#378ADD', '#7F77DD'])
axes[1].set_title('After SMOTE')
axes[1].set_ylabel('Count')
for i, v in enumerate([after_counts[0], after_counts[1]]):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

plt.suptitle('Effect of SMOTE on class distribution')
plt.tight_layout()
plt.show()

# ─── VISUALISE FEATURE SPACE (2D projection using two V features) ──────────
# Use V14 and V17 — the two strongest predictors from Phase 1 EDA

fig,axes = plt.subplots(1, 2, figsize=(14, 5))

# Before SMOTE - plot a sample so chart is readable
sample_before = pd.concat([
    X_train[y_train == 0].sample(500, random_state=42),
    X_train[y_train == 1 ]
])
labels_before = pd.concat([
    y_train[y_train == 0].sample(500, random_state=42),
    y_train[y_train == 1]
])

axes[0].scatter(sample_before.loc[labels_before==0, 'V14'],
                sample_before.loc[labels_before==0, 'V17'],
                c='#378ADD', alpha=0.4, s=20, label='Normal')
axes[0].scatter(sample_before.loc[labels_before==1, 'V14'],
                 sample_before.loc[labels_before==1, 'V17'],
                 c='#E24B4A', s=40, label='Fraud  (real)', zorder=5)
axes[0].set_title('Before SMOTE (V14 vs V17)')
axes[0].set_xlabel('V14'); axes[0].set_ylabel('V17')
axes[0].legend()

# After SMOTE
X_train_res_df = pd.DataFrame(X_train_res, columns=X_train.columns)
y_train_res_s  = pd.Series(y_train_res)

sample_after = pd.concat([
    X_train_res_df[y_train_res_s == 0].sample(500, random_state=42),
    X_train_res_df[y_train_res_s == 1].sample(800, random_state=42),
])
labels_after = pd.concat([
    y_train_res_s[y_train_res_s == 0].sample(500, random_state=42),
    y_train_res_s[y_train_res_s == 1].sample(800, random_state=42)
])

# Distinguish real vs synthetic fraud
real_fraud_idx = list(range(len(X_train)))   # Original indices
is_synthetic = labels_after == 1             # rough proxy for visualization

axes[1].scatter(sample_after.loc[labels_after==0, 'V14'],
                sample_after.loc[labels_after==0, 'V17'],
                c='#378ADD', alpha=0.4, s=20, label='Normal')
axes[1].scatter(sample_after.loc[labels_after==1, 'V14'],
                sample_after.loc[labels_after==1, 'V17'],
                c='#7F77DD', alpha=0.5, s=30, label='Fraud (synthetic)')
axes[1].set_title('After SMOTE (V14 vs V17)')
axes[1].set_xlabel('V14'); axes[1].set_ylabel('V17')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering Helper
# We work on the RESAMPLED training data and on X_test separately
# Never leak X_test values into feature engineering on training

X_train_fe = X_train_res_df.copy()
X_test_fe  = X_test.copy()

# ─── NEW FEATURE 1: Amount brackets ────────────────────────────────────────
# Fraud patterns differ by transaction size
# Small test charges (<= €1) are a known fraud tactic

def add_amount_features(df):
    # Note: we use 'scaled_amount' — the original Amount was dropped in Phase 2
    # But we can still create useful ratio and bucket features from it
    df['is_micro_txn'] = (df['scaled_amount'] < -0.35).astype(int)
    # scaled_amount < -0.35 corresponds roughly to Amount $1 after scaling
    return df

X_train_fe = add_amount_features(X_train_fe)
X_test_fe  = add_amount_features(X_test_fe)

print("New features 'is_micro_txn' added.")
print(f"Micro transactions in training:  {X_train_fe['is_micro_txn'].sum():,}")


# ─── NEW FEATURE 2: Interaction features ───────────────────────────────────
# V14 and V17 are individually strong — their product may be even stronger
# This captures non-linear relationships the model might miss on its own

def add_interaction_features(df):
    df['V14_x_V17'] = df['V14'] * df['V17']   # product interaction
    df['V14_x_V12'] = df['V14'] * df['V12']   # another strong pairing
    df['V17_x_V10'] = df['V17'] * df['V10']
    return df

X_train_fe = add_interaction_features(X_train_fe)
X_test_fe  = add_interaction_features(X_test_fe)

print("New features 'V14_x_V17' added.")
print(f"New column count:  {X_train_fe.shape[1]}")

# ─── NEW FEATURE 3: Magnitude features ─────────────────────────────────────
# Fraudsters often have extreme values in multiple V features simultaneously
# Capturing how "far from normal" a transaction is across key features

def add_magnitude_features(df):
    top_features = ['V14', 'V17', 'V12', 'V10', 'V11', 'V4']
    df['fraud_signal_magnitude'] = (
        df[top_features].abs().sum(axis=1)
    )
    return df

X_train_fe = add_magnitude_features(X_train_fe)
X_test_fe  = add_magnitude_features(X_test_fe)

print("Magnitude feature added.")


In [ ]:
# Verifying engineered features (sanity checks)
# Rebuild labels aligned with the feature-engineered training data
y_train_fe = pd.Series(y_train_res)

print("New features statistics by class:\n")
for feat in ['is_micro_txn', 'V14_x_V17', 'fraud_signal_magnitude']:
    normal_mean = X_train_fe.loc[y_train_fe == 0, feat].mean()
    fraud_mean  = X_train_fe.loc[y_train_fe == 1, feat].mean()
    ratio       = abs(fraud_mean / normal_mean) if normal_mean != 0 else float('inf')

    print(f"  {feat}:")
    print(f"   Normal mean:  {normal_mean:.2f}")
    print(f"   Fraud  mean:  {fraud_mean:.4f}")
    print(f"   Ratio:        {ratio:.2f}x  {'- strong signal' if ratio > 1.5 else ''}")
    print()

# A good engineered feature will have a meaningfully different mean
# between fraud and normal. If both means are nearly identical, drop it.

# Visualize Top Engineered  feature
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, feat in zip(axes, ['V14_x_V17', 'fraud_signal_magnitude', 'is_micro_txn']):
    normal_vals = X_train_fe.loc[y_train_fe == 0, feat].sample(2000, random_state=42)
    fraud_vals  = X_train_fe.loc[y_train_fe == 1, feat].sample(2000, random_state=42)

    ax.hist(normal_vals, bins=50, alpha=0.6, color='#378ADD',
            label='Normal', density=True)
    ax.hist(fraud_vals, bins=50, alpha=0.6, color='#E24B4A',
            label='Fraud', density=True)
    ax.set_title(f'{feat}')
    ax.set_xlabel('Feature value')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Engineered feature distributions - fraud vs normal')
plt.tight_layout()
plt.show()


In [ ]:
# Save SMOTE + Engineered features
# Save the final training data for Phase 4
import os, joblib

os.makedirs('phase3_output', exist_ok=True)

X_train_fe.to_csv('phase3_output/X_train_final.csv', index=False)
X_test_fe.to_csv( 'phase3_output/X_test_final.csv', index=False)
y_train_fe.to_csv('phase3_output/y_train_final.csv', index=False)
y_test.to_csv(   'phase3_output/y_test_final.csv', index=False)

print("Phase 3 output saved.")
print(f"Training set: {X_train_fe.shape[0]:,} rows,  {X_train_fe.shape[1]} features")
print(f"Test set:     {X_test_fe.shape[0]:,} rows")
print(f"Training fraud rate after SMOTE:  {y_train_fe.mean()*100:.1f}%")
print(f"Test fraud rate (real world):     {y_test.mean()*100:.4f}%")
